# Modelo municipal: K-means

Este notebook muestra primero el baseline del cliente: los cinco municipios con mas poblacion de 18 a 50 anos. Despues entrena K-means sobre la misma Gold local definitiva y exporta **785 filas** con las columnas `municipality_code` y `cluster`. Los candidatos, la facturacion, la viabilidad y la comparacion economica se calculan en negocio.

Entradas: `log1p(population_18_50)`, `population_growth_pct` y `mean_net_income_per_person`. El crecimiento es de **poblacion total**, no del segmento 18-50. No se utiliza ninguna variable de competencia.

Las rentas ausentes se imputan con la mediana observada, sin excluir municipios ni modificar Gold. Se aplica `StandardScaler`, `random_state=42` y `n_init=20`. Se evalua k=2..6 y se elige la mayor silhouette entre soluciones con al menos **30 municipios por cluster**; en empate se prefiere el menor k. Si ninguna cumple, el entrenamiento falla explicitamente.

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

working_directory = Path.cwd().resolve()
project_root = next(
    (path for path in (working_directory, *working_directory.parents)
     if (path / 'src' / 'modelo' / 'modelo.py').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Could not find the project root containing src/modelo.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.modelo.modelo import CLUSTERS_PATH, save_clusters, train_model
from src.negocio.valor_negocio import top5_baseline
from src.procesado.common.paths import GOLD_TABLE_PATH

logging.basicConfig(level=logging.INFO)


## 1. Baseline del cliente: los cinco municipios con mas poblacion objetivo

La decision anterior del cliente consiste en seleccionar los cinco municipios con mayor `population_18_50`, sin filtrar por competencia, crecimiento, renta ni viabilidad. Se utiliza la misma Gold que para K-means y la misma funcion de seleccion que en negocio. Esta tabla solo muestra la seleccion de referencia; no genera un fichero adicional ni calcula resultados economicos.

In [ ]:
gold = pd.read_parquet(GOLD_TABLE_PATH)
baseline = top5_baseline(gold).loc[:, [
    'municipality_code', 'municipality_name', 'province',
    'population_year', 'population_18_50',
]].reset_index(drop=True)
baseline.index = pd.RangeIndex(1, len(baseline) + 1, name='posicion')
display(baseline)


## 2. Gold: cobertura, nulos y periodos
La lectura y la exportacion son locales; este notebook no accede a S3.

In [ ]:
print(f'Gold: {len(gold)} municipios; origen: {GOLD_TABLE_PATH}')
display(gold.isna().sum().rename('missing_values').to_frame())
display(gold[['population_year', 'population_growth_start_year', 'income_year']].value_counts(dropna=False).rename('municipalities').reset_index())


## 3. Entrenamiento y seleccion de k
La inercia y silhouette describen la estructura de los datos, no la rentabilidad ni una mejora respecto al baseline.

In [ ]:
result = train_model(gold)
display(result.evaluation)
print(f'k seleccionado: {result.model.n_clusters}')
print(f'Mediana de renta para imputacion: {result.income_median:.2f}')
print(f'Municipios con renta imputada: {result.imputed_municipalities}')
display(result.profiles)


## 4. Exportacion para negocio
Unico fichero generado: `data/resultados/clusters.csv`. Las etiquetas son identificadores de grupo, no posiciones en un ranking. Al leer el CSV hay que indicar `dtype={'municipality_code': 'string'}` para conservar ceros iniciales.

In [ ]:
output_path = save_clusters(result.clusters)
restored = pd.read_csv(output_path, dtype={'municipality_code': 'string'})
pd.testing.assert_frame_equal(restored, result.clusters)
assert restored.shape == (785, 2)
assert restored['municipality_code'].is_unique
assert restored['municipality_code'].str.fullmatch(r'\d{5}').all()
print(f'Verificado: {restored.shape}, guardado en {CLUSTERS_PATH}')
display(restored.head(10))
